[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/34_speculative_decoding.ipynb)

# 🔴 Hard: Speculative Decoding

Implement the **acceptance/rejection step** of speculative decoding — a technique for accelerating LLM inference.

### Signature
```python
def speculative_decode(target_probs, draft_probs, draft_tokens) -> list[int]:
    # target_probs: (K, V) from target (large) model
    # draft_probs: (K, V) from draft (small) model
    # draft_tokens: (K,) tokens sampled by draft model
    # Returns: list of accepted tokens (1 to K)
```

### Algorithm
For each position i = 0, ..., K-1:
1. `ratio = target_probs[i, token_i] / draft_probs[i, token_i]`
2. Accept with probability `min(1, ratio)`
3. If rejected: sample from `normalize(max(0, target - draft))`, append, and stop

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.5 MB/s eta 0:00:00


In [2]:
import torch

In [50]:
# ✏️ YOUR IMPLEMENTATION HERE

def speculative_decode(target_probs, draft_probs, draft_tokens):

    n = len(draft_tokens)
    p_target = target_probs[torch.arange(n), draft_tokens]
    p_draft = draft_probs[torch.arange(n), draft_tokens]
    ratio = torch.clamp(p_target / p_draft, max = 1)
    p = torch.rand(n)
    keep = p <= ratio
    if sum(keep) < n:
      valid_prefix = torch.nonzero(~keep, as_tuple=True)[0][0].item()
      accept = draft_tokens[:valid_prefix]
      p_reject = torch.clamp(target_probs[valid_prefix] - draft_probs[valid_prefix], min = 0)
      p_reject = p_reject / torch.sum(p_reject)
      sampled_idx = torch.multinomial(p_reject, num_samples=1)

    elif sum(keep) == n:
      accept = draft_tokens
      sampled_idx = torch.multinomial(target_probs[-1], num_samples=1)

    out_token = torch.cat([accept, sampled_idx])

    return out_token

    # pass  # accept/reject loop

In [51]:
# 🧪 Debug
torch.manual_seed(0)
probs = torch.softmax(torch.randn(4, 10), dim=-1)
tokens = torch.tensor([2, 5, 1, 8])
print('Perfect draft:', speculative_decode(probs, probs, tokens))
target = torch.softmax(torch.randn(4, 10), dim=-1)
draft = torch.softmax(torch.randn(4, 10), dim=-1)
print('Random draft:', speculative_decode(target, draft, tokens))

Perfect draft: tensor([2, 5, 1, 8, 2])
Random draft: tensor([2, 5, 1, 8, 2])


In [52]:
# ✅ SUBMIT
from torch_judge import check
check('speculative_decoding')


🧪 Testing: Speculative Decoding (Hard)
──────────────────────────────────────────────────
  ❌ [1/3] Perfect draft: all accepted
     Perfect draft should accept all, got 5
  ✅ [2/3] Output length bounded (2.8ms)
  ✅ [3/3] All tokens valid (42.7ms)
──────────────────────────────────────────────────
  📊 2/3 tests passed.
  Keep going! Use hint("speculative_decoding") if you're stuck.

